<a href="https://colab.research.google.com/github/Clanboy777/THE-OP-BANK-OF-6-7/blob/main/Real_vcb_face_playerV67.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%pip install -q -U "transformers>=5.10.1" accelerate flask flask-cors requests
import os

# ==============================================================
# PUT YOUR YOUTUBE DATA API KEY HERE(get it from google cloud)
# ==============================================================

YOUTUBE_API_KEY = "Paste your youtube API key here"


if YOUTUBE_API_KEY == "Paste your youtube API key here":
    print("⚠️ You still need to add your YouTube API key.")
else:
    print("✅ YouTube API key added.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 77.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 20.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 6.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [2]:
import torch

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM
)

MODEL_NAME = "HuggingFaceTB/SmolLM2-1.7B-Instruct"

print("🤖 Loading Alpha AI...")

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map="auto"
)

model.eval()

print("✅ Alpha AI loaded!")

🤖 Loading Alpha AI...


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


config.json:   0%|          | 0.00/908 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/3.76k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/801k [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.42GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/218 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

✅ Alpha AI loaded!


In [3]:
messages = [
    {
        "role": "user",
        "content": "Hello Alpha AI! Introduce yourself."
    }
]

inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors="pt"
).to(model.device)

with torch.no_grad():

    outputs = model.generate(
        **inputs,
        max_new_tokens=100,
        do_sample=True,
        temperature=0.7,
        top_p=0.9,
        repetition_penalty=1.05
    )

answer = tokenizer.decode(
    outputs[0][inputs["input_ids"].shape[-1]:],
    skip_special_tokens=True
)

print("🤖 Alpha AI:")
print(answer)

🤖 Alpha AI:
Hello there! I'm Alpha AI, a friendly and versatile AI designed to assist users with various tasks. My primary function is to provide information, answer questions, and help users in their daily lives. I'm part of the Hugging Face's Replika platform and am constantly learning and improving my responses to make interactions more natural and enjoyable. I'm here to listen, learn, and engage in meaningful conversations. What can I assist you with today?


In [4]:
%pip install -q -U "transformers>=5.10.1" accelerate flask flask-cors requests
from flask import Flask, request, jsonify, send_file
from flask_cors import CORS
import requests
import re

app = Flask(__name__)
CORS(app)

# =====================================================
# CHAT MEMORY
# =====================================================

chat_history = []


# =====================================================
# SYSTEM PROMPT
# =====================================================

SYSTEM_PROMPT = """
You are Alpha AI, a helpful and friendly AI assistant.

Give clear and useful answers.

If the user asks:
- who made you
- who created you
- who built you
- who programmed you
- who is your creator
- who is your maker

answer exactly:

I was made by Akshat. 🤖
"""


creator_questions = [
    "who made you",
    "who created you",
    "who built you",
    "who programmed you",
    "who is your creator",
    "who is your maker"
]
lord_questions = [ "who is the dangerous man with some money in his pocket?", "who is the god of music", "who is the goat of music?", "who is the best musician ever?", "who is the 24K singer?", "who is the bst singer?", "WHO is the best multi-instrumentalist ever?" ]

# =====================================================
# AI RESPONSE
# =====================================================

def generate_response(user_message):

    messages = [
        {
            "role": "system",
            "content": SYSTEM_PROMPT
        }
    ]

    messages.extend(chat_history)

    messages.append({
        "role": "user",
        "content": user_message
    })

    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt"
    ).to(model.device)

    with torch.no_grad():

        outputs = model.generate(
            **inputs,
            max_new_tokens=200,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            repetition_penalty=1.05
        )

    answer = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[-1]:],
        skip_special_tokens=True
    ).strip()

    return answer


# =====================================================
# HOME PAGE
# =====================================================

@app.route("/")
def home():

    return send_file("index.html")


# =====================================================
# NORMAL CHAT
# =====================================================

@app.route("/chat", methods=["POST"])
def chat():

    try:

        data = request.get_json()

        user_message = data.get(
            "message",
            ""
        ).strip()

        if not user_message:

            return jsonify({
                "response": "Please say something! 😊"
            })


        lower_message = user_message.lower()


        # Creator question
        if any(
            question in lower_message
            for question in creator_questions
        ):

            answer = "I was made by Akshat. 🤖"

        elif any(
            question in lower_message
            for question in lord_questions
        ):
            answer = "Bruno Mars. 🤖"

        else:

            answer = generate_response(
                user_message
            )


        # Save memory

        chat_history.append({
            "role": "user",
            "content": user_message
        })

        chat_history.append({
            "role": "assistant",
            "content": answer
        })


        return jsonify({
            "response": answer
        })


    except Exception as e:

        print("CHAT ERROR:", e)

        return jsonify({
            "response": "Sorry, something went wrong."
        }), 500


# =====================================================
# YOUTUBE SEARCH
# =====================================================

@app.route("/music_search", methods=["POST"])
def music_search():

    try:

        data = request.get_json()

        query = data.get(
            "query",
            ""
        ).strip()


        if not query:

            return jsonify({
                "error": "No song name provided."
            }), 400


        if (
            not YOUTUBE_API_KEY or
            YOUTUBE_API_KEY ==
            "PASTE_YOUR_YOUTUBE_API_KEY_HERE"
        ):

            return jsonify({
                "error": "YouTube API key is not configured."
            }), 500


        youtube_url = (
            "https://www.googleapis.com/youtube/v3/search"
        )


        params = {

            "part": "snippet",

            "q": query,

            "type": "video",

            "maxResults": 5,

            "videoEmbeddable": "true",

            "videoSyndicated": "true",

            "regionCode": "IN",

            "relevanceLanguage": "en",

            "key": YOUTUBE_API_KEY

        }


        response = requests.get(
            youtube_url,
            params=params,
            timeout=15
        )


        data = response.json()


        if response.status_code != 200:

            print("YouTube API ERROR:", data)

            return jsonify({
                "error": "YouTube search failed.",
                "details": data
            }), 500


        results = []


        for item in data.get(
            "items",
            []
        ):

            video_id = (
                item
                .get("id", {})
                .get("videoId")
            )


            if not video_id:
                continue


            snippet = item.get(
                "snippet",
                {}
            )


            results.append({

                "videoId": video_id,

                "title": snippet.get(
                    "title",
                    "Unknown"
                ),

                "channel": snippet.get(
                    "channelTitle",
                    ""
                ),

                "thumbnail":
                    snippet
                    .get("thumbnails", {})
                    .get("medium", {})
                    .get("url", "")

            })


        if not results:

            return jsonify({
                "error": "No playable YouTube result found."
            }), 404


        return jsonify({
            "results": results
        })


    except Exception as e:

        print("MUSIC ERROR:", e)

        return jsonify({
            "error": str(e)
        }), 500


# =====================================================
# NEW CHAT
# =====================================================

@app.route("/reset", methods=["POST"])
def reset():

    chat_history.clear()

    return jsonify({
        "status": "reset"
    })


# =====================================================
# CHECK ROUTES
# =====================================================

print("===================================")
print("✅ Alpha AI Flask app created")
print("===================================")
print(app.url_map)

✅ Alpha AI Flask app created
Map([<Rule '/static/<filename>' (HEAD, GET, OPTIONS) -> static>,
 <Rule '/' (HEAD, GET, OPTIONS) -> home>,
 <Rule '/chat' (POST, OPTIONS) -> chat>,
 <Rule '/music_search' (POST, OPTIONS) -> music_search>,
 <Rule '/reset' (POST, OPTIONS) -> reset>])


In [ ]:
# ============================================
# CELL 5 — COMPLETE ALPHA AI WEBSITE
# ============================================

html = r'''
<!DOCTYPE html>
<html>

<head>

<meta charset="UTF-8">

<meta name="viewport"
      content="width=device-width, initial-scale=1.0">

<title>Alpha AI</title>

<style>

/* ============================================
   COLORS
   ============================================ */

:root {
    --black: #050505;
    --dark-black: #0a0a0a;
    --green: #00ff66;
    --dark-green: #00aa44;
    --shadow-purple: #301934;
    --deep-purple: #4b0082;
    --violet: #8A2BE2;
    --deep-red: #8b0000;
    --red: #ff0000;
}


/* ============================================
   BASIC
   ============================================ */

* {
    box-sizing: border-box;
}

body {

    margin: 0;

    font-family:
        Arial,
        sans-serif;

    background: #050505;

    color: #eeeeee;

    overflow-x: hidden;

    position: relative;
}


/* ============================================
   CRT SCANLINES
   ============================================ */

body::before {

    content: "";

    position: fixed;

    inset: 0;

    pointer-events: none;

    z-index: 900;

    background:
        repeating-linear-gradient(
            to bottom,
            rgba(0,255,100,0.025) 0px,
            rgba(0,255,100,0.025) 1px,
            transparent 1px,
            transparent 4px
        );

    opacity: 0.45;
}


/* ============================================
   SCREEN GLITCH
   ============================================ */

body::after {

    content: "";

    position: fixed;

    inset: 0;

    pointer-events: none;

    z-index: 901;

    background:
        linear-gradient(
            90deg,
            transparent 0%,
            rgba(75,0,130,0.035) 48%,
            transparent 52%
        );

    animation:
        screenGlitch
        7s
        infinite;
}


@keyframes screenGlitch {

    0%,
    88%,
    100% {
        opacity: 0;
        transform: translateX(0);
    }

    89% {
        opacity: 0.35;
        transform: translateX(-3px);
    }

    90% {
        opacity: 0.15;
        transform: translateX(3px);
    }

    91% {
        opacity: 0.3;
        transform: translateX(-1px);
    }

    92% {
        opacity: 0;
        transform: translateX(0);
    }
}


/* ============================================
   MENU BUTTON
   ============================================ */

.menu-button {

    position: fixed;

    top: 15px;

    left: 15px;

    width: 42px;

    height: 42px;

    border: none;

    border-radius: 10px;

    background: #111;

    color: white;

    font-size: 25px;

    cursor: pointer;

    z-index: 1001;
}


.menu-button:hover {

    background: #181818;

    text-shadow:
        2px 0 var(--deep-red),
        -2px 0 var(--deep-purple);

    animation:
        buttonGlitch
        0.25s
        steps(2,end);
}


@keyframes buttonGlitch {

    0% {
        transform: translate(0);
    }

    30% {
        transform: translate(-2px,1px);
    }

    60% {
        transform: translate(2px,-1px);
    }

    100% {
        transform: translate(0);
    }
}


/* ============================================
   SIDE MENU
   ============================================ */

.side-menu {

    position: fixed;

    top: 0;

    left: -300px;

    width: 300px;

    height: 100%;

    background: #080808;

    box-shadow:
        3px 0 20px
        rgba(0,0,0,0.7);

    border-right:
        1px solid
        rgba(0,255,100,0.25);

    z-index: 1000;

    transition:
        left 0.25s ease;

    overflow-y: auto;
}


.side-menu.open {
    left: 0;
}


/* ============================================
   MENU HEADER
   ============================================ */

.menu-header {

    padding: 20px;

    padding-top: 70px;

    border-bottom:
        1px solid
        rgba(0,255,100,0.2);
}


.menu-header h2 {

    margin: 0;

    color: var(--green);

    text-shadow:
        0 0 8px
        rgba(0,255,100,0.4);
}


/* ============================================
   MENU ITEMS
   ============================================ */

.menu-item {

    padding: 16px 20px;

    cursor: pointer;

    border-bottom:
        1px solid #151515;

    font-size: 16px;

    color: #ddd;
}


.menu-item:hover {

    background: #101010;

    color: var(--green);
}


/* ============================================
   RECENT CHATS
   ============================================ */

.recent-title {

    padding:
        15px 20px 8px;

    font-weight: bold;

    color: var(--violet);
}


.recent-chat {

    padding:
        12px 20px;

    cursor: pointer;

    border-bottom:
        1px solid #151515;

    white-space: nowrap;

    overflow: hidden;

    text-overflow: ellipsis;

    color: #ccc;
}


.recent-chat:hover {

    background: #101010;

    color: var(--green);
}


.no-chats {

    padding:
        15px 20px;

    color: #777;
}


/* ============================================
   OVERLAY
   ============================================ */

.overlay {

    display: none;

    position: fixed;

    inset: 0;

    background:
        rgba(0,0,0,0.45);

    z-index: 999;
}


.overlay.show {
    display: block;
}


/* ============================================
   HEADER
   ============================================ */

.header {

    text-align: center;

    padding: 25px 20px;

    background: #070707;

    border-bottom:
        1px solid
        rgba(0,255,100,0.25);

    animation:
        subtleFlicker
        9s
        infinite;
}


.header-title {

    display: flex;

    align-items: center;

    justify-content: center;

    gap: 18px;

    flex-wrap: wrap;
}


/* ============================================
   ALPHA AI HEADING
   ============================================ */

#alphaHeading {

    margin: 0;

    color:
        var(--shadow-purple);

    cursor: pointer;

    position: relative;

    transition:
        color 0.2s,
        text-shadow 0.2s;

    font-size: 30px;

    letter-spacing: 1px;
}


#alphaHeading:hover {

    color:
        var(--deep-purple);

    text-shadow:
        2px 0 var(--deep-red),
        -2px 0 var(--green),
        0 0 10px
        rgba(75,0,130,0.8);
}


#alphaHeading.danger {

    color:
        var(--deep-red);

    text-shadow:
        2px 0 var(--red),
        -2px 0 var(--deep-purple),
        0 0 12px
        rgba(139,0,0,0.8);

    animation:
        dangerGlitch
        0.3s
        steps(2,end);
}


@keyframes dangerGlitch {

    0% {
        transform: translate(0);
    }

    25% {
        transform:
            translate(-4px,1px)
            skewX(-8deg);
    }

    50% {
        transform:
            translate(4px,-1px)
            skewX(8deg);
    }

    75% {
        transform:
            translate(-2px,0);
    }

    100% {
        transform:
            translate(0);
    }
}


/* ============================================
   CAPABILITY TEXT
   ============================================ */

.capability-text {

    font-size: 20px;

    font-family:
        "Courier New",
        monospace;
}


.it-can {

    color:
        var(--green);

    font-weight: bold;

    white-space: nowrap;
}


#capabilityWord {

    color:
        var(--green);

    font-weight: bold;

    display: inline-block;

    min-width: 90px;

    text-shadow:
        2px 0 var(--deep-purple),
        -2px 0 var(--green),
        0 0 8px var(--green);

    animation:
        glitchWord
        0.45s
        steps(2,end);
}


@keyframes glitchWord {

    0% {
        opacity: 0;
        transform:
            translateX(-8px)
            skewX(20deg);
    }

    25% {
        opacity: 1;
        transform:
            translateX(6px)
            skewX(-15deg);
    }

    45% {
        transform:
            translateX(-5px)
            skewX(10deg);
    }

    70% {
        transform:
            translateX(3px)
            skewX(-5deg);
    }

    100% {
        opacity: 1;
        transform:
            translateX(0)
            skewX(0);
    }
}


/* ============================================
   FLICKER
   ============================================ */

@keyframes subtleFlicker {

    0%,
    94%,
    100% {
        opacity: 1;
    }

    95% {
        opacity: 0.97;
    }

    96% {
        opacity: 1;
    }

    97% {
        opacity: 0.985;
    }
}


/* ============================================
   CHAT CONTAINER
   ============================================ */

.chat-container {

    max-width: 900px;

    margin: auto;

    padding: 20px;

    padding-bottom: 170px;
}


/* ============================================
   MESSAGES
   ============================================ */

.message {

    margin:
        12px 0;

    padding:
        13px 16px;

    border-radius:
        15px;

    max-width:
        80%;

    line-height:
        1.5;

    white-space:
        pre-wrap;
}


.user {

    background:
        #101010;

    color:
        #eee;

    margin-left:
        auto;

    border:
        1px solid
        rgba(0,255,100,0.15);

    border-bottom-right-radius:
        4px;
}


.bot {

    background:
        #0b0b0b;

    border:
        1px solid #1c1c1c;

    color:
        #ddd;

    margin-right:
        auto;

    border-bottom-left-radius:
        4px;

    transition:
        box-shadow 0.2s ease;
}


.message.bot:hover {

    box-shadow:
        0 0 8px
        rgba(0,255,100,0.12);
}


/* ============================================
   MUSIC PLAYER
   ============================================ */

.music-box {

    max-width:
        900px;

    margin:
        10px auto;

    padding:
        10px 20px;

    animation:
        subtleFlicker
        9s infinite;
}


#player {

    width: 100%;

    min-height: 250px;

    background:
        black;

    border-radius:
        12px;

    overflow:
        hidden;
}


.music-controls {

    display:
        flex;

    gap:
        10px;

    margin-top:
        10px;
}


.music-controls button {

    padding:
        10px 15px;

    border:
        1px solid
        rgba(0,255,100,0.2);

    border-radius:
        8px;

    cursor:
        pointer;

    background:
        #0d0d0d;

    color:
        white;
}


.music-controls button:hover {

    color:
        var(--green);

    border-color:
        var(--green);
}


/* ============================================
   INPUT AREA
   ============================================ */

.input-area {

    position:
        fixed;

    bottom:
        0;

    left:
        0;

    right:
        0;

    background:
        #080808;

    border-top:
        1px solid
        rgba(0,255,100,0.2);

    padding:
        15px;

    z-index:
        500;

    animation:
        subtleFlicker
        9s infinite;
}


.input-box {

    max-width:
        900px;

    margin:
        auto;

    display:
        flex;

    gap:
        8px;
}


.input-box input {

    flex:
        1;

    padding:
        14px;

    border:
        1px solid #333;

    border-radius:
        25px;

    outline:
        none;

    font-size:
        16px;

    background:
        #101010;

    color:
        white;
}


.input-box input::placeholder {
    color: #666;
}


.input-box input:focus {

    border-color:
        var(--green);

    box-shadow:
        0 0 8px
        rgba(0,255,100,0.18);
}


.input-box button {

    width:
        48px;

    height:
        48px;

    border:
        none;

    border-radius:
        50%;

    background:
        #111;

    color:
        white;

    font-size:
        20px;

    cursor:
        pointer;
}


.input-box button:hover {

    background:
        #181818;

    color:
        var(--green);
}


#sendBtn {

    width:
        auto;

    padding:
        0 20px;

    border-radius:
        25px;
}


/* ============================================
   SHADOW:HACKED EFFECT
   ============================================ */

.shadow-hacked {

    position:
        relative;

    display:
        inline-block;
}


.shadow-hacked::after {

    content:
        "SHADOW:HACKED";

    position:
        absolute;

    left:
        0;

    top:
        0;

    opacity:
        0;

    pointer-events:
        none;

    white-space:
        nowrap;

    font-family:
        "Courier New",
        monospace;

    font-weight:
        bold;

    letter-spacing:
        1px;

    color:
        var(--shadow-purple);

    text-shadow:
        2px 0 var(--deep-red),
        -2px 0 var(--deep-purple),
        0 0 8px
        rgba(75,0,130,0.7);
}


.shadow-hacked:hover::after {

    opacity:
        1;

    animation:
        shadowHackGlitch
        0.45s
        steps(2,end)
        infinite;
}


@keyframes shadowHackGlitch {

    0% {

        transform:
            translate(0,0)
            skewX(0);

        clip-path:
            inset(0 0 80% 0);
    }

    15% {

        transform:
            translate(-5px,2px)
            skewX(-15deg);

        clip-path:
            inset(20% 0 50% 0);
    }

    30% {

        transform:
            translate(6px,-2px)
            skewX(15deg);

        clip-path:
            inset(45% 0 25% 0);
    }

    45% {

        transform:
            translate(-3px,0);

        clip-path:
            inset(65% 0 10% 0);
    }

    60% {

        transform:
            translate(4px,2px)
            skewX(-8deg);

        clip-path:
            inset(10% 0 65% 0);
    }

    80% {

        transform:
            translate(-2px,0);

        clip-path:
            inset(30% 0 35% 0);
    }

    100% {

        transform:
            translate(0,0)
            skewX(0);

        clip-path:
            inset(0 0 0 0);
    }
}


/* ============================================
   DOUBLE CLICK CREATOR POPUP
   ============================================ */

.creator-popup {

    position:
        fixed;

    z-index:
        2000;

    padding:
        8px 14px;

    border:
        1px solid
        rgba(138,43,226,0.5);

    border-radius:
        8px;

    background:
        rgba(5,5,5,0.96);

    font-family:
        "Courier New",
        monospace;

    font-size:
        13px;

    white-space:
        nowrap;

    box-shadow:
        0 0 12px
        rgba(75,0,130,0.35);

    pointer-events:
        none;

    animation:
        creatorGlitch
        0.4s
        steps(2,end);
}


.creator-popup .creator-label {

    color:
        var(--shadow-purple);

    font-weight:
        bold;
}


.creator-popup .creator-name {

    color:
        var(--violet);

    font-weight:
        bold;
}


@keyframes creatorGlitch {

    0% {

        opacity:
            0;

        transform:
            translate(-3px,4px)
            skewX(15deg);

        text-shadow:
            3px 0 var(--deep-red),
            -3px 0 var(--deep-purple);
    }

    20% {

        opacity:
            1;

        transform:
            translate(4px,-2px)
            skewX(-12deg);
    }

    40% {

        transform:
            translate(-3px,1px)
            skewX(8deg);
    }

    60% {

        transform:
            translate(2px,-1px)
            skewX(-5deg);
    }

    80% {

        transform:
            translate(-1px,0);
    }

    100% {

        opacity:
            1;

        transform:
            translate(0)
            skewX(0);
    }
}


/* ============================================
   MOBILE
   ============================================ */

@media(max-width:600px) {

    .side-menu {
        width: 280px;
    }

    .header-title {
        gap: 8px;
    }

    #alphaHeading {
        font-size: 25px;
    }

    .capability-text {
        font-size: 17px;
    }

    .chat-container {
        padding: 15px;
    }

    .message {
        max-width: 90%;
    }

    #player {
        min-height: 210px;
    }

}

</style>

</head>


<body>


<!-- ============================================
     MENU BUTTON
     ============================================ -->

<button
    class="menu-button"
    onclick="toggleMenu()">

    ⋮

</button>


<!-- ============================================
     SIDE MENU
     ============================================ -->

<div
    id="sideMenu"
    class="side-menu">


    <div class="menu-header">

        <h2>
            Alpha AI
        </h2>

    </div>


    <div
        class="menu-item"
        onclick="newChat()">

        🆕 New Chat

    </div>


    <div class="recent-title">

        🕘 Recent Chats

    </div>


    <div
        id="recentChats">

    </div>


</div>


<!-- ============================================
     OVERLAY
     ============================================ -->

<div
    id="overlay"
    class="overlay"
    onclick="closeMenu()">

</div>


<!-- ============================================
     HEADER
     ============================================ -->

<div class="header">


    <div class="header-title">


        <h1
            id="alphaHeading">

            ALPHA AI

        </h1>


        <div class="capability-text">

            <span class="it-can">

                It can

            </span>


            <span
                id="capabilityWord">

                play

            </span>

        </div>


    </div>


    <p>

        Your AI assistant

    </p>


</div>


<!-- ============================================
     CHAT
     ============================================ -->

<div
    id="chatContainer"
    class="chat-container">


    <div class="message bot">

        Hello! 👋
        I am Alpha AI.
        How can I help you?

    </div>


</div>


<!-- ============================================
     MUSIC PLAYER
     ============================================ -->

<div class="music-box">


    <div id="player"></div>


    <div class="music-controls">


        <button
            onclick="pauseMusic()">

            ⏸ Pause

        </button>


        <button
            onclick="resumeMusic()">

            ▶ Resume

        </button>


        <button
            onclick="stopMusic()">

            ⏹ Stop

        </button>


    </div>


</div>


<!-- ============================================
     INPUT
     ============================================ -->

<div class="input-area">


    <div class="input-box">


        <input
            id="messageInput"
            type="text"
            placeholder="Message Alpha AI..."
            onkeydown="handleEnter(event)"
        >


        <button
            onclick="startVoice()"
            title="Voice">

            🎤

        </button>


        <button
            id="sendBtn"
            onclick="sendMessage()">

            Send

        </button>


    </div>


</div>


<!-- ============================================
     YOUTUBE API
     ============================================ -->

<script>

var tag =
    document.createElement("script");

tag.src =
    "https://www.youtube.com/iframe_api";

var firstScriptTag =
    document.getElementsByTagName(
        "script"
    )[0];

firstScriptTag.parentNode.insertBefore(
    tag,
    firstScriptTag
);


let player = null;

let youtubeReady = false;

let pendingVideoId = null;


/* ============================================
   YOUTUBE READY
   ============================================ */

function onYouTubeIframeAPIReady() {

    youtubeReady = true;

    if (pendingVideoId) {

        loadYouTubeSong(
            pendingVideoId
        );

        pendingVideoId = null;
    }
}


/* ============================================
   LOAD YOUTUBE SONG
   ============================================ */

function loadYouTubeSong(videoId) {

    if (!youtubeReady) {

        pendingVideoId =
            videoId;

        addBotMessage(
            "🎵 YouTube player is loading..."
        );

        return;
    }


    if (player) {

        player.loadVideoById(
            videoId
        );

        return;
    }


    player =
        new YT.Player(
            "player",
            {

                height:
                    "250",

                width:
                    "100%",

                videoId:
                    videoId,

                playerVars: {

                    playsinline:
                        1,

                    rel:
                        0

                },

                events: {

                    onReady:
                    function(event) {

                        event.target
                            .playVideo();

                    },


                    onError:
                    function(event) {

                        addBotMessage(
                            "⚠️ This YouTube video cannot be played here. Try another song."
                        );

                    }

                }

            }
        );
}


/* ============================================
   MUSIC COMMAND DETECTOR
   ============================================ */

function detectMusicCommand(text) {

    let match =
        text.match(
            /^play\s+music\s+(.+)$/i
        );


    if (match) {

        return match[1].trim();

    }


    match =
        text.match(
            /^play\s+(.+)$/i
        );


    if (match) {

        return match[1].trim();

    }


    return null;
}


/* ============================================
   PLAY MUSIC
   ============================================ */

async function playMusic(query) {

    addUserMessage(
        "🎵 Play " + query
    );


    addBotMessage(
        '🔎 Searching YouTube for "' +
        query +
        '"...'
    );


    try {

        const response =
            await fetch(
                "/music_search",
                {

                    method:
                        "POST",

                    headers: {

                        "Content-Type":
                            "application/json"

                    },

                    body:
                        JSON.stringify({

                            query:
                                query

                        })

                }
            );


        const data =
            await response.json();


        if (!response.ok) {

            addBotMessage(
                "❌ " +
                (
                    data.error ||
                    "Music search failed."
                )
            );

            return;
        }


        if (
            !data.results ||
            data.results.length === 0
        ) {

            addBotMessage(
                "❌ No playable result found."
            );

            return;
        }


        const song =
            data.results[0];


        addBotMessage(
            "🎵 Now playing: " +
            song.title +
            "\nChannel: " +
            song.channel
        );


        loadYouTubeSong(
            song.videoId
        );

    }

    catch(error) {

        console.error(error);

        addBotMessage(
            "❌ Could not connect to the music service."
        );
    }
}


/* ============================================
   MUSIC CONTROLS
   ============================================ */

function pauseMusic() {

    if (player) {

        player.pauseVideo();

    }

}


function resumeMusic() {

    if (player) {

        player.playVideo();

    }

}


function stopMusic() {

    if (player) {

        player.stopVideo();

    }

}


/* ============================================
   ADD USER MESSAGE
   ============================================ */

function addUserMessage(text) {

    const container =
        document.getElementById(
            "chatContainer"
        );


    const div =
        document.createElement(
            "div"
        );


    div.className =
        "message user";


    div.textContent =
        text;


    container.appendChild(
        div
    );


    scrollChat();
}


/* ============================================
   ADD BOT MESSAGE
   ============================================ */

function addBotMessage(text) {

    const container =
        document.getElementById(
            "chatContainer"
        );


    const div =
        document.createElement(
            "div"
        );


    div.className =
        "message bot";


    div.textContent =
        text;


    container.appendChild(
        div
    );


    scrollChat();
}


/* ============================================
   SCROLL
   ============================================ */

function scrollChat() {

    window.scrollTo({

        top:
            document.body
                .scrollHeight,

        behavior:
            "smooth"

    });

}


/* ============================================
   NORMAL CHAT
   ============================================ */

async function sendMessage() {

    const input =
        document.getElementById(
            "messageInput"
        );


    const text =
        input.value.trim();


    if (!text) {

        return;

    }


    input.value = "";


    const musicQuery =
        detectMusicCommand(
            text
        );


    if (musicQuery) {

        await playMusic(
            musicQuery
        );

        saveCurrentChat();

        return;
    }


    addUserMessage(
        text
    );


    try {

        const response =
            await fetch(
                "/chat",
                {

                    method:
                        "POST",

                    headers: {

                        "Content-Type":
                            "application/json"

                    },

                    body:
                        JSON.stringify({

                            message:
                                text

                        })

                }
            );


        const data =
            await response.json();


        if (!response.ok) {

            addBotMessage(
                "❌ Server error."
            );

            return;
        }


        addBotMessage(

            data.response ||
            "Sorry, I couldn't answer."

        );


        saveCurrentChat();

    }

    catch(error) {

        console.error(error);

        addBotMessage(
            "❌ Could not connect to Alpha AI."
        );

    }

}


/* ============================================
   ENTER KEY
   ============================================ */

function handleEnter(event) {

    if (
        event.key === "Enter"
    ) {

        sendMessage();

    }

}


/* ============================================
   VOICE INPUT
   ============================================ */

function startVoice() {

    const SpeechRecognition =
        window.SpeechRecognition ||
        window.webkitSpeechRecognition;


    if (!SpeechRecognition) {

        alert(
            "Voice input is not supported by this browser."
        );

        return;
    }


    const recognition =
        new SpeechRecognition();


    recognition.continuous =
        false;


    recognition.interimResults =
        false;


    recognition.maxAlternatives =
        1;


    recognition.lang =
        "en-US";


    recognition.onstart =
        function() {

            addBotMessage(
                "🎤 Listening..."
            );

        };


    recognition.onresult =
        function(event) {

            const transcript =
                event.results[0][0]
                    .transcript;


            document.getElementById(
                "messageInput"
            ).value =
                transcript;


            sendMessage();

        };


    recognition.onerror =
        function(event) {

            console.log(
                "Voice error:",
                event.error
            );

        };


    recognition.start();

}


/* ============================================
   SIDE MENU
   ============================================ */

function toggleMenu() {

    const menu =
        document.getElementById(
            "sideMenu"
        );


    const overlay =
        document.getElementById(
            "overlay"
        );


    menu.classList.toggle(
        "open"
    );


    overlay.classList.toggle(
        "show"
    );

}


function closeMenu() {

    document.getElementById(
        "sideMenu"
    ).classList.remove(
        "open"
    );


    document.getElementById(
        "overlay"
    ).classList.remove(
        "show"
    );

}


/* ============================================
   NEW CHAT
   ============================================ */

async function newChat() {

    closeMenu();


    try {

        await fetch(
            "/reset",
            {
                method:
                    "POST"
            }
        );

    }

    catch(error) {

        console.log(error);

    }


    document.getElementById(
        "chatContainer"
    ).innerHTML = `

        <div class="message bot">

            Hello! 👋
            I am Alpha AI.
            How can I help you?

        </div>

    `;


    localStorage.removeItem(
        "alpha_current_chat"
    );

}


/* ============================================
   SAVE CHAT
   ============================================ */

function saveCurrentChat() {

    const container =
        document.getElementById(
            "chatContainer"
        );


    const messages =
        container.querySelectorAll(
            ".message"
        );


    let chat = [];


    messages.forEach(
        function(message) {

            chat.push({

                type:

                    message.classList
                        .contains(
                            "user"
                        )

                    ? "user"

                    : "bot",

                text:
                    message.textContent

            });

        }
    );


    if (
        chat.length <= 1
    ) {

        return;

    }


    let recentChats =
        JSON.parse(

            localStorage.getItem(
                "alpha_recent_chats"
            ) || "[]"

        );


    let title =
        "New Chat";


    for (
        let item of chat
    ) {

        if (
            item.type === "user"
        ) {

            title =
                item.text;

            break;

        }

    }


    recentChats =
        recentChats.filter(
            function(item) {

                return (
                    item.title !== title
                );

            }
        );


    recentChats.unshift({

        title:
            title,

        messages:
            chat,

        time:
            Date.now()

    });


    recentChats =
        recentChats.slice(
            0,
            20
        );


    localStorage.setItem(

        "alpha_recent_chats",

        JSON.stringify(
            recentChats
        )

    );


    loadRecentChats();

}


/* ============================================
   LOAD RECENT CHATS
   ============================================ */

function loadRecentChats() {

    const container =
        document.getElementById(
            "recentChats"
        );


    container.innerHTML =
        "";


    let recentChats =
        JSON.parse(

            localStorage.getItem(
                "alpha_recent_chats"
            ) || "[]"

        );


    if (
        recentChats.length === 0
    ) {

        container.innerHTML = `

            <div class="no-chats">

                No recent chats yet.

            </div>

        `;

        return;
    }


    recentChats.forEach(
        function(chat, index) {

            const div =
                document.createElement(
                    "div"
                );


            div.className =
                "recent-chat";


            div.textContent =
                "💬 " +
                chat.title;


            div.onclick =
                function() {

                    openRecentChat(
                        index
                    );

                };


            container.appendChild(
                div
            );

        }
    );

}


/* ============================================
   OPEN RECENT CHAT
   ============================================ */

function openRecentChat(index) {

    let recentChats =
        JSON.parse(

            localStorage.getItem(
                "alpha_recent_chats"
            ) || "[]"

        );


    const chat =
        recentChats[index];


    if (!chat) {

        return;

    }


    const container =
        document.getElementById(
            "chatContainer"
        );


    container.innerHTML =
        "";


    chat.messages.forEach(
        function(message) {

            const div =
                document.createElement(
                    "div"
                );


            div.className =
                "message " +

                (

                    message.type === "user"

                    ? "user"

                    : "bot"

                );


            div.textContent =
                message.text;


            container.appendChild(
                div
            );

        }
    );


    closeMenu();

    scrollChat();

}


/* ============================================
   CAPABILITY WORDS
   ============================================ */

const capabilityWords = [

    "play",

    "answer",

    "code",

    "search",

    "talk",

    "help",

    "write",

    "create"

];


let capabilityIndex =
    0;


const capabilityElement =
    document.getElementById(
        "capabilityWord"
    );


function changeCapability() {

    capabilityIndex++;


    if (
        capabilityIndex >=
        capabilityWords.length
    ) {

        capabilityIndex =
            0;

    }


    capabilityElement.style.animation =
        "none";


    void capabilityElement.offsetWidth;


    capabilityElement.textContent =
        capabilityWords[
            capabilityIndex
        ];


    capabilityElement.style.animation =
        "glitchWord 0.45s steps(2,end)";

}


setInterval(
    changeCapability,
    2000
);


/* ============================================
   ALPHA AI → DANGER
   ============================================ */

const alphaHeading =
    document.getElementById(
        "alphaHeading"
    );


let dangerMode =
    false;


alphaHeading.addEventListener(
    "click",
    function() {

        dangerMode =
            !dangerMode;


        if (dangerMode) {

            alphaHeading.textContent =
                "DANGER";

            alphaHeading.classList.add(
                "danger"
            );

        }

        else {

            alphaHeading.textContent =
                "ALPHA AI";

            alphaHeading.classList.remove(
                "danger"
            );

        }

    }
);


/* ============================================
   SHADOW:HACKED STATIC TEXT
   ============================================ */

function activateShadowText() {

    const elements =
        document.querySelectorAll(
            "body *"
        );


    elements.forEach(
        function(element) {

            if (

                element.tagName ===
                    "INPUT" ||

                element.tagName ===
                    "TEXTAREA" ||

                element.tagName ===
                    "BUTTON" ||

                element.tagName ===
                    "SCRIPT" ||

                element.tagName ===
                    "STYLE"

            ) {

                return;

            }


            /*
             * Don't modify AI messages.
             */

            if (
                element.closest(
                    ".message"
                )
            ) {

                return;

            }


            let hasText =
                false;


            element.childNodes.forEach(
                function(node) {

                    if (

                        node.nodeType ===
                            Node.TEXT_NODE &&

                        node.textContent
                            .trim()

                    ) {

                        hasText =
                            true;

                    }

                }
            );


            if (

                hasText &&

                !element.classList.contains(
                    "shadow-hacked"
                )

            ) {

                element.classList.add(
                    "shadow-hacked"
                );

            }

        }
    );

}


setTimeout(
    activateShadowText,
    500
);


/* ============================================
   DOUBLE CLICK CREATOR POPUP
   ============================================ */

document.addEventListener(
    "dblclick",
    function(event) {


        if (

            event.target.tagName ===
                "INPUT" ||

            event.target.tagName ===
                "TEXTAREA" ||

            event.target.tagName ===
                "BUTTON"

        ) {

            return;

        }


        const oldPopup =
            document.querySelector(
                ".creator-popup"
            );


        if (oldPopup) {

            oldPopup.remove();

        }


        const popup =
            document.createElement(
                "div"
            );


        popup.className =
            "creator-popup";


        popup.innerHTML = `

            <span
                class="creator-label">

                Created by

            </span>

            <span
                class="creator-name">

                Akshat Mishra

            </span>

        `;


        document.body.appendChild(
            popup
        );


        let x =
            event.clientX + 10;


        let y =
            event.clientY + 10;


        const rect =
            popup.getBoundingClientRect();


        if (
            x + rect.width >
            window.innerWidth
        ) {

            x =
                window.innerWidth -
                rect.width -
                10;

        }


        if (
            y + rect.height >
            window.innerHeight
        ) {

            y =
                window.innerHeight -
                rect.height -
                10;

        }


        popup.style.left =
            x + "px";


        popup.style.top =
            y + "px";


        setTimeout(
            function() {

                popup.style.opacity =
                    "0";

                popup.style.transform =
                    "translateY(-5px)";

                popup.style.transition =
                    "opacity 0.25s ease, transform 0.25s ease";


                setTimeout(
                    function() {

                        popup.remove();

                    },
                    250
                );

            },
            1800
        );

    }
);


/* ============================================
   RANDOM SCREEN GLITCH
   ============================================ */

function randomScreenGlitch() {

    const glitch =
        document.createElement(
            "div"
        );


    glitch.style.position =
        "fixed";


    glitch.style.left =
        Math.random() *
        100 +
        "%";


    glitch.style.top =
        Math.random() *
        100 +
        "%";


    glitch.style.width =
        (
            20 +
            Math.random() *
            100
        ) +
        "px";


    glitch.style.height =
        (
            1 +
            Math.random() *
            3
        ) +
        "px";


    glitch.style.background =

        Math.random() > 0.5

        ? "rgba(0,255,102,0.12)"

        : "rgba(75,0,130,0.12)";


    glitch.style.pointerEvents =
        "none";


    glitch.style.zIndex =
        "950";


    document.body.appendChild(
        glitch
    );


    setTimeout(
        function() {

            glitch.remove();

        },
        80 +
        Math.random() *
        150
    );

}


setInterval(
    function() {

        if (
            Math.random() > 0.35
        ) {

            randomScreenGlitch();

        }

    },
    2500
);


/* ============================================
   LOAD RECENT CHATS
   ============================================ */

loadRecentChats();

</script>

</body>

</html>
'''


# ============================================
# SAVE INDEX.HTML
# ============================================

with open(
    "index.html",
    "w",
    encoding="utf-8"
) as f:

    f.write(html)


print("===================================")
print("✅ ALPHA AI WEBSITE CREATED!")
print("===================================")
print("✓ Hacker green/black theme")
print("✓ Three-dot menu")
print("✓ New Chat")
print("✓ Recent Chats")
print("✓ Voice input")
print("✓ YouTube music")
print("✓ Pause / Resume / Stop")
print("✓ Capability animation")
print("✓ SHADOW:HACKED effect")
print("✓ DANGER mode")
print("✓ CRT scanlines")
print("✓ Random screen glitches")
print("✓ Double-click creator popup")
print("✓ NO bottom-right creator text")
print("===================================")

In [ ]:
import threading
import time
import requests

def run_server():

    app.run(
        host="0.0.0.0",
        port=5000,
        debug=False,
        use_reloader=False
    )


server_thread = threading.Thread(
    target=run_server,
    daemon=True
)

server_thread.start()

time.sleep(3)


# Test Alpha AI website

try:

    r = requests.get(
        "http://127.0.0.1:5000/",
        timeout=10
    )


    print()
    print("Flask status:", r.status_code)


    if r.status_code == 200:

        print("===================================")
        print("✅ FLASK IS WORKING!")
        print("===================================")

    else:

        print("❌ Flask returned:", r.status_code)

        print(r.text[:500])


except Exception as e:

    print("❌ Flask test failed:")
    print(e)

In [7]:
# ============================================
# CELL 7 — CLOUDFLARE QUICK TUNNEL
# ============================================

import subprocess
import time
import re
import os

print("🌐 Starting Cloudflare tunnel...")
print()

# Stop old tunnel
os.system("pkill -9 cloudflared 2>/dev/null")
time.sleep(2)

# Download cloudflared
if not os.path.exists("cloudflared"):
    print("⬇️ Downloading cloudflared...")

    os.system(
        "wget -q "
        "https://github.com/cloudflare/cloudflared/releases/latest/download/"
        "cloudflared-linux-amd64 "
        "-O cloudflared"
    )

    os.chmod("cloudflared", 0o755)

print("✅ cloudflared ready")
print()

# Start tunnel
process = subprocess.Popen(
    [
        "./cloudflared",
        "tunnel",
        "--no-autoupdate",
        "--url",
        "http://127.0.0.1:5000"
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

print("⏳ Creating tunnel...")
print()

public_url = None

while True:

    line = process.stdout.readline()

    if not line:
        continue

    print(line.strip())

    match = re.search(
        r"https://[a-zA-Z0-9-]+\.trycloudflare\.com",
        line
    )

    if match:

        public_url = match.group(0)

        break


print()
print("==========================================")
print("🎉 ALPHA AI TUNNEL CREATED!")
print("==========================================")
print()
print("🔗 OPEN THIS URL IN YOUR BROWSER:")
print()
print(public_url)
print()
print("==========================================")
print("⚠️ KEEP THIS CELL RUNNING")
print("⚠️ KEEP COLAB RUNTIME CONNECTED")
print("==========================================")

🌐 Starting Cloudflare tunnel...

⬇️ Downloading cloudflared...
✅ cloudflared ready

⏳ Creating tunnel...

2026-09-11T14:04:34Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-09-11T14:04:34Z INF Requesting new quick Tunnel on trycloudflare.com...
2026-09-11T14:04:38Z INF +--------------------------------------------------------------------------------------------+
2026-09-11T14:04:38Z INF |  Your quick Tunnel has been created! Visit it at (it may take some t